# VDB2 빌드 — 표 벡터DB

audit_tables.db에 description, embedding_text 열 추가 → ChromaDB

In [ ]:
!pip install chromadb -q

In [1]:
import sqlite3
import shutil
import ollama
import chromadb
from tqdm import tqdm

In [2]:
AUDIT_DB_PATH  = "./audit_tables.db"
GEMINI_DB_PATH = "./gemini_tables.db"
CHROMA_PATH    = "./chroma_db"
COLLECTION     = "vdb2_table"
EMBED_MODEL    = "qwen3-embedding:8b"
BATCH_SIZE     = 8

## 1. audit_tables.db에 description, embedding_text 열 추가

In [3]:
conn_a = sqlite3.connect(AUDIT_DB_PATH)
conn_g = sqlite3.connect(GEMINI_DB_PATH)
cur_a  = conn_a.cursor()
cur_g  = conn_g.cursor()

# 열이 없을 때만 추가
existing = [r[1] for r in cur_a.execute("PRAGMA table_info(tables)").fetchall()]
if "description" not in existing:
    cur_a.execute("ALTER TABLE tables ADD COLUMN description TEXT")
    print("description 열 추가")
if "embedding_text" not in existing:
    cur_a.execute("ALTER TABLE tables ADD COLUMN embedding_text TEXT")
    print("embedding_text 열 추가")

# gemini DB에서 description 가져와서 순서 기준으로 매칭
cur_g.execute("SELECT description FROM tables ORDER BY year, table_id")
descriptions = [r[0] for r in cur_g.fetchall()]

cur_a.execute("SELECT rowid, title, row_headers_md, column_headers_md FROM tables ORDER BY year, table_id")
audit_rows = cur_a.fetchall()

assert len(descriptions) == len(audit_rows), "행수 불일치!"


def build_embedding_text(title, row_headers_md, column_headers_md, description):
    parts = []
    if title and title.strip():
        parts.append(f"제목: {title.strip()}")
    if row_headers_md and row_headers_md.strip():
        items = [r.strip('| ').strip() for r in row_headers_md.split('\n')
                 if r.strip() and '---' not in r and 'row_header' not in r]
        if items:
            parts.append("행 항목: " + ", ".join(items[:20]))
    if column_headers_md and column_headers_md.strip():
        items = [r.strip('| ').strip() for r in column_headers_md.split('\n')
                 if r.strip() and '---' not in r and 'column_header' not in r]
        if items:
            parts.append("열 항목: " + ", ".join(items[:10]))
    if description and description.strip():
        parts.append(description.strip())
    return "\n".join(parts)


for (rowid, title, row_headers_md, col_headers_md), desc in zip(audit_rows, descriptions):
    emb_text = build_embedding_text(title, row_headers_md, col_headers_md, desc)
    cur_a.execute(
        "UPDATE tables SET description=?, embedding_text=? WHERE rowid=?",
        (desc, emb_text, rowid)
    )

conn_a.commit()
conn_a.close()
conn_g.close()
print(f"완료 — {len(audit_rows)}개 행 업데이트")

# 샘플 확인
conn_a = sqlite3.connect(AUDIT_DB_PATH)
cur_a  = conn_a.cursor()
cur_a.execute("SELECT table_id, embedding_text FROM tables LIMIT 1")
r = cur_a.fetchone()
print(f"\n[{r[0]}] embedding_text 샘플:\n{r[1][:400]}")
conn_a.close()

완료 — 1308개 행 업데이트

[samsung_2014_table_0001] embedding_text 샘플:
제목: 2015년 2월 24일
행 항목: 이 감사보고서는 감사보고서일(2015년2월 24일) 현재로 유효한 것입니다. 따라서 감사보고서일 이후 이 보고서를 열람하는 시점까지의 기간 사이에 회사의 재무제표에 중대한 영향을 미칠 수 있는 사건이나 상황이 발생할 수도 있으며 이로 인하여 이 감사보고서가 수정될 수도 있습니다.
## 1. 이 표가 무엇을 나타내는지 자세히 설명

이 표는 삼성전자 2014년 감사보고서에 포함된, 감사보고서의 유효성과 관련된 중요한 고지사항을 담고 있습니다. 주요 내용은 해당 감사보고서가 특정 감사보고서일(2015년 2월 24일) 현재의 정보만을 반영하며, 그 이후에 발생할 수 있는 중대한 사건이나 상황으로 인해 회사의 재무제표가 영향을 받고 감사보고서가 수정될 가능성이 있음을 명확히


## 2. ChromaDB 컬렉션 생성 및 임베딩

In [4]:
client = chromadb.PersistentClient(path=CHROMA_PATH)
try:
    client.delete_collection(COLLECTION)
    print("기존 컬렉션 삭제")
except:
    pass
collection = client.create_collection(
    name=COLLECTION,
    metadata={"hnsw:space": "cosine"},
)
print(f"컬렉션 '{COLLECTION}' 생성")

# audit_tables.db에서 embedding_text 로드
conn_a = sqlite3.connect(AUDIT_DB_PATH)
cur_a  = conn_a.cursor()
cur_a.execute("""
    SELECT table_id, year, title, footnotes, embedding_text
    FROM tables
    WHERE embedding_text IS NOT NULL
    ORDER BY year, table_id
""")
rows = cur_a.fetchall()
conn_a.close()
print(f"{len(rows)}개 행 로드")


컬렉션 'vdb2_table' 생성
1308개 행 로드


In [7]:
for i in tqdm(range(0, len(rows), BATCH_SIZE), desc="임베딩 중"):
    batch = rows[i : i + BATCH_SIZE]

    ids        = [r[0] for r in batch]
    documents  = [r[4] for r in batch]   # embedding_text
    metadatas  = [
        {
            "year":      r[1],
            "title":     r[2] or "",
            "footnotes": r[3] or "",
        }
        for r in batch
    ]

    embeddings = ollama.embed(model=EMBED_MODEL, input=documents).embeddings

    collection.add(
        ids=ids,
        documents=documents,
        embeddings=embeddings,
        metadatas=metadatas,
    )

print(f"완료 — 총 {collection.count()}개 벡터 저장")

## 3. 검색 테스트

VDB2에서 table_id 반환 → audit_tables.db에서 markdown, footnotes 조회

In [ ]:
def search_tables(query: str, year: int, p: int = 2):
    query_emb = ollama.embed(model=EMBED_MODEL, input=[query]).embeddings

    results = collection.query(
        query_embeddings=query_emb,
        n_results=p,
        where={"year": year},
    )

    table_ids = results["ids"][0]
    conn_a = sqlite3.connect(AUDIT_DB_PATH)
    cur_a  = conn_a.cursor()
    placeholders = ",".join("?" * len(table_ids))
    cur_a.execute(
        f"SELECT table_id, title, markdown, footnotes FROM tables WHERE table_id IN ({placeholders})",
        table_ids,
    )
    table_data = {r[0]: r for r in cur_a.fetchall()}
    conn_a.close()

    for tid, meta, dist in zip(table_ids, results["metadatas"][0], results["distances"][0]):
        row = table_data[tid]
        print(f"[{meta['year']}년] {row[1]} (score: {1-dist:.3f})")
        print(f"  markdown: {row[2][:150]}...")
        if row[3]:
            print(f"  footnotes: {row[3][:80]}")
        print()


search_tables("매출액 영업이익", year=2023)
print("="*60)
search_tables("재고자산 평가충당금", year=2022)